## 🎯 Learning Objectives
* Understand the core components of a ReAct loop: Thought, Action, and Observation.
* Implement a basic ReAct loop structure in Python to simulate agent reasoning and interaction.
* Simulate the interaction between an agent, a mock tool, and an environment.
* Evaluate an agent's ability to generate coherent thoughts and execute actions based on observations.


## AG03-L02: Building the ReAct loop: Thought, Action, Observation

### Exercise Task

In this exercise, you will implement a simplified ReAct (Reasoning and Acting) loop from scratch. The goal is to understand the fundamental mechanics of how an agent can iteratively generate thoughts, choose actions, execute those actions using tools, and process observations to make progress towards a goal.

Your agent will be given a `goal`. It will then enter a loop where it:
1. **Thinks**: Generates a textual thought based on the current goal and any previous observations.
2. **Acts**: Decides on an action to take, which could be using a predefined tool or signaling completion.
3. **Observes**: Receives an observation from the environment or tool execution.

This cycle repeats until the agent decides it has achieved its goal or a maximum number of steps is reached.

### Requirements

1.  **Implement a `MockSearchTool` class**: This class should simulate a simple search engine. It will have a `run` method that takes a `query` (string) and returns a predefined, mock `observation` (string).
2.  **Implement an `Agent` class**: This class will encapsulate the ReAct loop logic.
    *   It should have an `__init__` method that takes a list of available `tools` (e.g., an instance of `MockSearchTool`).
    *   It must have a `run` method that takes a `goal` (string) and an optional `max_steps` (integer, default to 5).
    *   Inside the `run` method, implement the ReAct loop:
        *   Maintain a history of thoughts, actions, and observations.
        *   **Thought Generation**: For simplicity, your agent's thought can be a simple string indicating its current reasoning based on the goal and the last observation.
        *   **Action Selection**: The agent should decide whether to use the `MockSearchTool` or issue a `FINISH` action. For this exercise, a simple heuristic is sufficient (e.g., if the goal contains a keyword, search for it; otherwise, finish).
        *   **Action Execution & Observation**: If a tool action is selected, execute the tool's `run` method and capture its output as the `observation`. If `FINISH` is selected, terminate the loop.
        *   The loop should terminate if `FINISH` is called or `max_steps` is reached.
3.  **Output**: The `run` method should return a list of dictionaries, where each dictionary represents a step in the ReAct loop, containing `{'step': int, 'thought': str, 'action': str, 'observation': str}`.

### Evaluation Criteria

*   **Correct ReAct Loop Implementation**: The `Agent.run` method correctly follows the Thought -> Action -> Observation cycle.
*   **Tool Integration**: The `Agent` successfully utilizes the `MockSearchTool`.
*   **Reasonable Logic**: The agent's thought generation and action selection, while simplified, demonstrate a basic form of reasoning towards the goal.
*   **Termination Conditions**: The loop correctly terminates upon `FINISH` action or `max_steps` limit.
*   **Code Quality**: Code is clean, well-commented, and adheres to Python best practices.


In [ ]:
# Setup Code: Mock Tool Definitions

class MockSearchTool:
    """A mock search tool that simulates searching for information."""
    def __init__(self, name="SearchTool"):
        self.name = name
        self.description = "A tool for searching information on the web."

    def run(self, query: str) -> str:
        """Simulates a search query and returns a predefined observation."""
        print(f"  -> Executing {self.name} with query: '{query}'")
        # Simulate different search results based on query keywords
        if "AgenticLabs" in query or "AI Agents" in query:
            return "Observation: AgenticLabs.ng is a leading platform for building advanced AI agents and automation tools. They offer courses like AG-03 on building agents from scratch."
        elif "ReAct loop" in query or "Thought Action Observation" in query:
            return "Observation: The ReAct (Reasoning and Acting) framework combines reasoning traces with task-specific actions. It involves an iterative cycle of Thought, Action, and Observation."
        elif "Python" in query:
            return "Observation: Python is a high-level, general-purpose programming language. Its design philosophy emphasizes code readability with the use of significant indentation."
        else:
            return f"Observation: No highly relevant results found for '{query}'. Try a different query."

# Example of how to instantiate and use the tool
# search_tool = MockSearchTool()
# result = search_tool.run("What is AgenticLabs?")
# print(result)


### Your Turn: Implement the ReAct Agent

Now, implement the `Agent` class according to the requirements. Focus on making the ReAct loop clear and functional. You can use the `MockSearchTool` defined above.

```python
# Define your Agent class here

class Agent:
    def __init__(self, tools: list):
        # Initialize the agent with available tools
        pass

    def run(self, goal: str, max_steps: int = 5) -> list:
        # Implement the ReAct loop
        pass

# Example Usage (after implementing your Agent class):
# search_tool_instance = MockSearchTool()
# my_agent = Agent(tools=[search_tool_instance])
# 
# print("\n--- Running Agent for Goal 1 ---")
# history_1 = my_agent.run("Find information about AgenticLabs.ng and its courses.")
# for step in history_1:
#     print(f"\nStep {step['step']}:")
#     print(f"  Thought: {step['thought']}")
#     print(f"  Action: {step['action']}")
#     print(f"  Observation: {step['observation']}")
# 
# print("\n--- Running Agent for Goal 2 ---")
# history_2 = my_agent.run("Explain the ReAct loop concept.")
# for step in history_2:
#     print(f"\nStep {step['step']}:")
#     print(f"  Thought: {step['thought']}")
#     print(f"  Action: {step['action']}")
#     print(f"  Observation: {step['observation']}")
```


In [ ]:
# Reference Solution: ReAct Agent Implementation

class Agent:
    """A simple ReAct agent that iteratively thinks, acts, and observes."""

    def __init__(self, tools: list):
        """Initializes the agent with a list of available tools."""
        self.tools = {tool.name: tool for tool in tools}
        self.history = [] # To store the trace of the ReAct loop

    def _generate_thought(self, goal: str, last_observation: str) -> str:
        """Generates a simple thought based on the goal and last observation.
        In a real agent, this would be powered by an LLM.
        """
        if not last_observation:
            return f"I need to start by understanding the goal: '{goal}'. I should search for relevant information."
        elif "No highly relevant results" in last_observation:
            return f"My last search for '{self.history[-1]['action_args']['query']}' yielded no strong results. I need to refine my approach or conclude if no further action is possible."
        elif "AgenticLabs" in last_observation and "courses" in goal:
            return f"I found information about AgenticLabs.ng and its courses. This seems to address the goal. I should conclude."
        elif "ReAct (Reasoning and Acting) framework" in last_observation and "ReAct loop concept" in goal:
            return f"I have found a good explanation of the ReAct loop. I can now conclude."
        else:
            return f"I have observed: '{last_observation}'. I need to decide my next step to achieve the goal: '{goal}'."

    def _select_action(self, goal: str, thought: str) -> tuple[str, dict]:
        """Selects an action (tool use or FINISH) based on the thought and goal.
        In a real agent, this would be powered by an LLM.
        """
        if "conclude" in thought.lower() or "no further action" in thought.lower():
            return "FINISH", {"result": "Goal achieved or no further progress possible."}

        # Simple heuristic for action selection
        if "AgenticLabs" in goal and "AgenticLabs" not in thought:
            return "SearchTool", {"query": "AgenticLabs.ng courses"}
        elif "ReAct loop" in goal and "ReAct (Reasoning and Acting) framework" not in thought:
            return "SearchTool", {"query": "ReAct loop concept"}
        elif "find information" in goal and "search" in thought.lower():
            # Generic search if specific keywords not matched yet
            return "SearchTool", {"query": goal.replace("find information about ", "").replace("explain ", "").strip()}
        
        return "FINISH", {"result": "Could not determine a clear action, finishing."}

    def run(self, goal: str, max_steps: int = 5) -> list:
        """Executes the ReAct loop to achieve the given goal."""
        print(f"\n--- Starting Agent for Goal: '{goal}' ---")
        self.history = []
        last_observation = ""

        for step_num in range(1, max_steps + 1):
            print(f"\n--- Step {step_num} ---")

            # 1. Thought
            thought = self._generate_thought(goal, last_observation)
            print(f"Thought: {thought}")

            # 2. Action
            action_type, action_args = self._select_action(goal, thought)
            action_str = f"{action_type}({', '.join([f'{k}={repr(v)}' for k, v in action_args.items()])})"
            print(f"Action: {action_str}")

            current_observation = ""
            if action_type == "FINISH":
                current_observation = action_args.get("result", "Agent decided to finish.")
                print(f"Observation: {current_observation}")
                self.history.append({
                    'step': step_num,
                    'thought': thought,
                    'action': action_str,
                    'observation': current_observation
                })
                print(f"--- Agent Finished. Final Observation: {current_observation} ---")
                break
            elif action_type in self.tools:
                tool = self.tools[action_type]
                try:
                    current_observation = tool.run(**action_args)
                except TypeError as e:
                    current_observation = f"Error: Invalid arguments for tool {action_type}: {e}"
                except Exception as e:
                    current_observation = f"Error executing tool {action_type}: {e}"
            else:
                current_observation = f"Error: Unknown action type '{action_type}'."
            
            print(f"Observation: {current_observation}")

            self.history.append({
                'step': step_num,
                'thought': thought,
                'action': action_str,
                'observation': current_observation
            })
            last_observation = current_observation
        else:
            print(f"--- Agent reached max steps ({max_steps}) without finishing. ---")

        return self.history

# --- Example Usage of the Reference Solution ---

# Instantiate the mock tool
search_tool_instance = MockSearchTool()

# Create an agent with the tool
my_agent = Agent(tools=[search_tool_instance])

# Run the agent for a specific goal
print("\n=====================================================")
print("Running Agent for Goal 1: Find information about AgenticLabs.ng and its courses.")
print("=====================================================")
agent_history_1 = my_agent.run("Find information about AgenticLabs.ng and its courses.", max_steps=3)

# Print the full history for review
print("\n--- Full History for Goal 1 ---")
for entry in agent_history_1:
    print(f"Step {entry['step']}:")
    print(f"  Thought: {entry['thought']}")
    print(f"  Action: {entry['action']}")
    print(f"  Observation: {entry['observation']}")
    print("--------------------------------------------------")

print("\n=====================================================")
print("Running Agent for Goal 2: Explain the ReAct loop concept.")
print("=====================================================")
agent_history_2 = my_agent.run("Explain the ReAct loop concept.", max_steps=3)

# Print the full history for review
print("\n--- Full History for Goal 2 ---")
for entry in agent_history_2:
    print(f"Step {entry['step']}:")
    print(f"  Thought: {entry['thought']}")
    print(f"  Action: {entry['action']}")
    print(f"  Observation: {entry['observation']}")
    print("--------------------------------------------------")

print("\n=====================================================")
print("Running Agent for Goal 3: Search for something obscure.")
print("=====================================================")
agent_history_3 = my_agent.run("Search for something obscure like 'quantum entanglement of socks'.", max_steps=2)

# Print the full history for review
print("\n--- Full History for Goal 3 ---")
for entry in agent_history_3:
    print(f"Step {entry['step']}:")
    print(f"  Thought: {entry['thought']}")
    print(f"  Action: {entry['action']}")
    print(f"  Observation: {entry['observation']}")
    print("--------------------------------------------------")
